In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.data.load_data import load_ratings
from src.data.train_test_split import time_split

ratings = load_ratings()
train, test = time_split(ratings)

In [3]:
ratings = ratings.sample(5_000_000, random_state=42)

We use a 5M sample of the MovieLens dataset for efficient experimentation and model comparison. All models are evaluated on the same subset to ensure fair comparison.

In [6]:
len(test)

4000053

In [4]:
# 1. Global Mean RMSE
import numpy as np
from sklearn.metrics import mean_squared_error

global_mean = train["rating"].mean()

y_true = test["rating"].values
y_pred = np.full(len(test), global_mean)

rmse_global = np.sqrt(mean_squared_error(y_true, y_pred))

print("Global Mean RMSE:", rmse_global)


Global Mean RMSE: 1.0200113541123996


Baseline model predicts the global mean rating for all users and items. It represents the simplest possible prediction without personalization.
Model performance is evaluated by comparing RMSE/MAE against this baseline, not against the mean value itself.

📌 Insight: If you always give ALL users a rating of 3.51, you'll be off by about 1 rating on average (in the RMSE metric)

In [5]:
# 2. User Bias Model RMSE
user_mean = train.groupby("userId")["rating"].mean()

global_mean = train["rating"].mean()

def predict_user(row):
    user = row["userId"]
    return user_mean.get(user, global_mean)

y_true = test["rating"].values
y_pred = test.apply(predict_user, axis=1)

rmse_user = np.sqrt(mean_squared_error(y_true, y_pred))

print("User Bias RMSE:", rmse_user)

User Bias RMSE: 1.0131716646896232


📌 User Bias model shows only a small improvement over Global Mean.
This suggests that user-level effects are weak in this dataset compared to item-level effects.

In [7]:
# 3. Item Bias Model RMSE
item_mean = train.groupby("movieId")["rating"].mean()

global_mean = train["rating"].mean()

def predict_item(row):
    item = row["movieId"]
    return item_mean.get(item, global_mean)

y_pred = test.apply(predict_item, axis=1)

rmse_item = np.sqrt(mean_squared_error(test["rating"], y_pred))

print("Item Bias RMSE:", rmse_item)

Item Bias RMSE: 0.9475375381531892


📌Insight:  The item bias model significantly outperforms the user bias model.   
Movie-specific effects have a stronger influence on ratings than user-specific tendencies   
This suggests that item popularity or inherent quality plays a major role in user ratings   

🎯 Implication   
Models should prioritize learning item representations   
This supports the use of matrix factorization methods, which capture item latent features effectively

In [8]:
global_mean = train["rating"].mean()

user_bias = train.groupby("userId")["rating"].mean() - global_mean
item_bias = train.groupby("movieId")["rating"].mean() - global_mean

def predict(row):
    u = row["userId"]
    i = row["movieId"]
    
    bu = user_bias.get(u, 0)
    bi = item_bias.get(i, 0)
    
    return global_mean + bu + bi

preds = test.apply(predict, axis=1)
y_true = test["rating"].values

rmse_value = np.sqrt(mean_squared_error(y_true, preds))
print("User + Item Bias RMSE:", rmse_value)


User + Item Bias RMSE: 0.9421589928757717



📌 Insight: Limited gain from adding user bias
Adding user bias to the item bias model results in only a marginal improvement in RMSE.

🎯 Implication
The dataset is more influenced by item characteristics than user preferences
Further improvements require modeling user-item interactions, not just independent biases